Se decidio usar 5 años como tiempo de estudio para poder estudiar el cambio y la evolucion de la calidad de el agua.
en esos 5 años podemos ver modificaciones, ciclos, cambios, sequias, humedad, epocas de mucha lluvia, inundaciones, etc que paso el agua considero, que es un cantidad  intermedia para crear un dataset sintetico 
POdemos tener informacion historia suficiente para diferenciar y ver los cambios que sufre el agua en estos 5 ciclos.

Ademas para nuestra RNN es una beuna cantidad de datos para que no se olvide de los mas antiguos, sino inente mantener la mayor catidad de datos para su 

**5 años para poder predecir el siguiente año
2021 a 2025 para obtener 2026.**
seria 365 datos diarios * 5 años=1.825 * 3 tomas por dia= 5,475 datos en total

In [4]:
import pandas as pd
import numpy as np
np.random.seed(42)


### Cantidad de muestras 

Al ser un dataset sintetico y sencillo, se puede tomar prinipalemtne 3 muestras al dia diciendo en los harios importante, mañana,tarde y noche.

Tenemos que tener infomracion temporal nuestro en este caso tomaremos en cuenta fecha y hora como Timestam, es decir a que hora y fecha se tomo la muestra.
Como extra para control y organizaciond el dataset año mes y dia.

In [5]:

fecha_inicio = "2021-01-01"
fecha_fin = "2025-12-31"

fechas = pd.date_range(
    start=fecha_inicio,
    end=fecha_fin,
    freq="D"
)

horas = ["06:00", "12:00", "20:00" ]

datos = []

for fecha in fechas:
    for hora in horas:
        datos.append([fecha, hora])

df = pd.DataFrame(datos, columns=["fecha", "hora"])

print(df.head())
print(df.shape)

       fecha   hora
0 2021-01-01  06:00
1 2021-01-01  12:00
2 2021-01-01  20:00
3 2021-01-02  06:00
4 2021-01-02  12:00
(5478, 2)


### Limitantes a tomar en cuenta

La scarectiristicas a usar tiene un numero limite de calculo
- ph = 6,5 -9.0
	- turbidez  =< 5
	-  solidos disueltos SDT =< 1000
	- color =<15
	- Nitrato (No3) =< 45
	- Arsenicos =< 0.1
	- Plomo <= 0.0
	- Ecoli =< 1; ausencia

Puede existir variuaciones de que nos sirve tener un dataset que siempre cumple, tambie un objetivo es poder ver las anomalias, al ser informacion de la naturaleza puede variar mucho, no podemos predecir que siempre peude pasar.

Es importante tomar en cuenta las estaciones del año, ya que hay epocas que llueve mas, menos o nada.
tambien usar valores del agua que son estandares pero con logica con algunas fluctiuaciones o anomalias en algunos casos.

In [6]:
df = pd.DataFrame(
    datos,
    columns=["fecha", "hora"]
)

In [7]:
df.shape

(5478, 2)

In [8]:
df.head()

,fecha,hora
0,2021-01-01,06:00
1,2021-01-01,12:00
2,2021-01-01,20:00
3,2021-01-02,06:00
4,2021-01-02,12:00


In [10]:
print("total registris", len(df))

total registris 5478


In [11]:
n = len(df)

### pH 6.5 - 9.0
tomar valores promedios de los datos dados 6.5 + 9.0 = 15,5 / 2 = 7,75 y como valor menimo 1.5

In [12]:
df["ph"] = np.random.normal(
    7.8,
    1.5,
    n
)

### Turbidez  <= 5
se hara lo mismo, se tomara un promedio y como minimo de variacion tmb.
limite <= 5

In [14]:
df["turbidez"] = np.random.normal(
    2.5,
    1.2,
    n
)

### Solidos disueltos <=1000

In [15]:
df["sdt"] = np.random.normal(
    550,
    150,
    n
)

### Color <=15

In [16]:
df["color"] = np.random.normal(
    7,
    3,
    n
)

### Nitrato (NO3) <= 45

In [17]:
df["nitrato"] = np.random.normal(
    23,
    10,
    n
)

### Arsenico <= 0.01

In [18]:
df["arsenico"] = np.random.normal(
    0.04,
    0.025,
    n
)

No permitir valores negativos, porque no tiene sentido.


In [19]:
df["turbidez"] = df["turbidez"].clip(lower=0)

df["sdt"] = df["sdt"].clip(lower=0)

df["color"] = df["color"].clip(lower=0)

df["nitrato"] = df["nitrato"].clip(lower=0)

df["arsenico"] = df["arsenico"].clip(lower=0)

### Plomo =< 0

In [20]:
df["plomo"] = np.random.choice(
    [0, 0.01, 0.02],
    size=n,
    p=[0.95, 0.03, 0.02]
)

### Ecoli , ausencia= 0 ;prese <1

In [21]:
df["ecoli"] = np.random.choice(
    [0, 1, 2, 3],
    size=n,
    p=[0.94, 0.03, 0.02, 0.01]
)

### Estaciones del año

In [22]:
df["mes"] = df["fecha"].dt.month

Se aumentaran ciertos mese scon mas lluvia para que sea mas realisata el dataset, por tanto, aumetnara tambien la turbidez y el color del agua


In [23]:
lluvia = df["mes"].isin(
    [11, 12, 1, 2, 3]
)

In [24]:
df.loc[lluvia, "turbidez"] += 1
df.loc[lluvia, "color"] += 2

In [25]:
print(df.head())

       fecha   hora        ph  turbidez         sdt      color    nitrato  \
0 2021-01-01  06:00  7.158843  1.000000  321.483691  12.536172  30.867986   
1 2021-01-01  12:00  7.949492  3.769304  577.048187  10.462853  24.868362   
2 2021-01-01  20:00  8.379919  3.919538  650.284392   6.793408  23.308519   
3 2021-01-02  06:00  7.625307  3.660199  332.781284  11.992263  37.861636   
4 2021-01-02  12:00  7.441386  1.000000  519.417480   8.374522   0.393997   

   arsenico  plomo  ecoli  mes  
0  0.031124    0.0      0    1  
1  0.022041    0.0      0    1  
2  0.002417    0.0      0    1  
3  0.041211    0.0      0    1  
4  0.020086    0.0      0    1  


### Variables para la calidad de los parametros del agua

Para el pH

In [26]:
def calidad_ph(ph):

    if 6.5 <= ph <= 9:
        return 10

    elif 6 <= ph < 6.5 or 9 < ph <= 9.5:
        return 7

    elif 5.5 <= ph < 6 or 9.5 < ph <= 10:
        return 4

    else:
        return 0

In [27]:
df["calidad_ph"] = df["ph"].apply(
    calidad_ph
)

Hacemos lo mismo para cada parametro poniendo los limites 

In [28]:
def calidad_limite(valor, limite):

    if valor <= limite * 0.5:
        return 10

    elif valor <= limite * 0.75:
        return 8

    elif valor <= limite:
        return 6

    elif valor <= limite * 1.25:
        return 3

    else:
        return 0

In [29]:
df["calidad_turbidez"] = df["turbidez"].apply(
    lambda x: calidad_limite(x, 5)
)

In [31]:
df["calidad_sdt"] = df["sdt"].apply(
    lambda x: calidad_limite(x, 1000)
)

In [ ]:
df["calidad_color"] = df["color"].apply(
    lambda x: calidad_limite(x, 15)
)

In [ ]:
df["calidad_nitrato"] = df["nitrato"].apply(
    lambda x: calidad_limite(x, 45)
)

In [34]:
df["calidad_arsenico"] = df["arsenico"].apply(
    lambda x: calidad_limite(x, 0.1)
)

In [36]:
df["calidad_plomo"] = df["plomo"].apply(
    lambda x: 10 if x == 0 else 0
)

In [40]:
df["calidad_ecoli"] = df["ecoli"].apply(
    lambda x: 10 if x == 0 else 0
)

sE ANADE LAS COLJUMANS DE CALIDAD DE CADA PARAMTERO

In [41]:
columnas_calidad = [
    "calidad_ph",
    "calidad_turbidez",
    "calidad_sdt",
    "calidad_color",
    "calidad_nitrato",
    "calidad_arsenico",
    "calidad_plomo",
    "calidad_ecoli"
]

In [42]:
df["suma_calidad"] = df[
    columnas_calidad
].sum(axis=1)

KeyError: "['calidad_color', 'calidad_nitrato'] not in index"